# Full Code Comparison: pre-refactor vs refactored `allele_freq.py`

Runs both pipelines end-to-end on MRGM_1066 and asserts equivalence of all three output tables.

In [17]:
import gc
import logging
import os
import shutil
import sys
import tempfile
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.home() / 'AlleleFlux-dev'))

import alleleflux.scripts.analysis._allele_freq_common as _cm
from alleleflux.scripts.analysis._allele_freq_common import (
    NUCLEOTIDES,
    CACHE_COLUMNS_LONGITUDINAL,
    build_metadata_from_qc,
    calculate_frequencies,
    load_qc_results,
    process_mag_files,
)
from alleleflux.scripts.analysis.allele_freq import (
    calculate_allele_frequency_changes,
    filter_constant_positions,
    get_mean_change,
    load_cache_files,
)
from alleleflux.scripts.analysis.allele_freq_cache import filter_qc_to_timepoint
from alleleflux.scripts.utilities.logging_config import setup_logging

setup_logging()
logging.getLogger().setLevel(logging.WARNING)

print('Imports OK')
print(f'NUCLEOTIDES: {NUCLEOTIDES}')

Imports OK
NUCLEOTIDES: ['A_frequency', 'T_frequency', 'G_frequency', 'C_frequency']


In [18]:
SORT_CHANGES = ['subjectID', 'contig', 'position', 'gene_id', 'replicate', 'group']
SORT_MEAN    = ['contig', 'position', 'gene_id', 'replicate', 'group']


def compare_outputs(old_df, new_df, sort_key, label, atol=1e-12):
    print(f'\n\u2500\u2500 {label} \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500')
    print(f'  Old: {old_df.shape}   New: {new_df.shape}')
    if 'MAG_ID' in old_df.columns and 'MAG_ID' not in new_df.columns:
        old_df = old_df.drop(columns=['MAG_ID'])
    old_cols, new_cols = set(old_df.columns), set(new_df.columns)
    if old_cols != new_cols:
        print(f'  [WARN] Only in old: {sorted(old_cols - new_cols)}')
        print(f'         Only in new: {sorted(new_cols - old_cols)}')
        common = sorted(old_cols & new_cols)
        old_df, new_df = old_df[common], new_df[common]
    key = [k for k in sort_key if k in old_df.columns]
    old_s = old_df.sort_values(key).reset_index(drop=True)
    new_s = new_df.sort_values(key).reset_index(drop=True)
    if len(old_s) != len(new_s):
        print(f'  [FAIL] Row count mismatch: old={len(old_s):,} new={len(new_s):,}')
        return False
    try:
        pd.testing.assert_frame_equal(old_s, new_s, check_like=False, check_dtype=False, atol=atol, rtol=0)
        print(f'  [PASS] {len(old_s):,} rows match exactly (atol={atol})')
        return True
    except AssertionError:
        print('  [FAIL]')
        for col in old_s.select_dtypes(include='number').columns:
            diff = (old_s[col] - new_s[col]).abs()
            if (diff > atol).any():
                print(f'    {col}: {(diff > atol).sum():,} rows differ (max delta={diff.max():.2e})')
        for col in old_s.select_dtypes(exclude='number').columns:
            bad = (old_s[col].astype(str) != new_s[col].astype(str)).sum()
            if bad:
                print(f'    {col}: {bad:,} rows differ')
        return False


print('Helpers defined.')

Helpers defined.


In [19]:
MAG_ID    = 'MRGM_1066'
DATA_TYPE = 'longitudinal'
TIMEPOINTS = ('5mo', '10mo')  # tp1 first — sets order for new code's unique()
GR_COMBO  = '1D_AL'
GROUPS    = ['1D', 'AL']

SCRATCH  = '/scratch/gpfs/AMOELLER/sidd/drido'
OUTDIR   = f'{SCRATCH}/alleleflux_output/alleleflux_output_1/longitudinal'
QC_DIR   = f'{OUTDIR}/QC/QC_5mo_10mo-{GR_COMBO}'
QC_FILE  = f'{QC_DIR}/{MAG_ID}_QC.tsv'

assert os.path.exists(QC_FILE), f'QC file not found: {QC_FILE}'

qc_full = pd.read_csv(QC_FILE, sep='\t')
passing = qc_full[qc_full['coverage_threshold_passed']]
print(f'{MAG_ID}: {len(passing)} passing samples')
print(passing[['sample_id', 'group', 'subjectID', 'replicate', 'time']].to_string(index=False))

MRGM_1066: 3 passing samples
      sample_id group  subjectID  replicate time
DO_AL_0140_021w    AL DO-AL-0140 DO-AL-0140  5mo
DO_AL_0142_044w    AL DO-AL-0142 DO-AL-0142 10mo
DO_AL_0140_044w    AL DO-AL-0140 DO-AL-0140 10mo


---
## Full Code Comparison — verbatim old `allele_freq.py` vs new

This section pastes the **complete pre-refactor `allele_freq.py`** into the notebook,
runs both pipelines end-to-end on MRGM_1066, and compares all three output files.

### What we are testing

| Step | Old code | New code |
|---|---|---|
| **Load QC** | `load_qc_results` (reads TSV, filters passed) | identical — shared from `_allele_freq_common` |
| **Read profiles** | `process_mag_files` via `Pool` (keeps extra cols: `MAG_ID`, `breadth`, `genome_size`, ...) | `process_mag_files` (drops extra cols immediately via `CACHE_COLUMNS_LONGITUDINAL`) |
| **Timepoint order** | `unique_timepoints = set()` → **non-deterministic** unpack | `allele_df["time"].unique()` — **first-occurrence** from cache concat |
| **Diff algorithm** | per-subject `pd.merge` loop (N merges) | one global `pd.merge` with `subjectID` in the key (1 merge) |
| **Write `_changes`** | `.tsv.gz` (slow — Python-level gzip) | `.parquet` Snappy (5–10× faster) |
| **Also writes** | `_allele_frequency_longitudinal.tsv.gz` | nothing (Parquet cache replaces it) |
| **`filter_constant_positions`** | unchanged | unchanged — identical code |
| **`get_mean_change`** | unchanged | unchanged — identical code |

### The set() bug

The old code did:
```python
unique_timepoints = set()
for subject_data in data_dict.values():
    unique_timepoints.update(subject_data.keys())
timepoint_1, timepoint_2 = unique_timepoints   # unpacking a set — no guaranteed order
```
With Python hash randomisation (`PYTHONHASHSEED`), `timepoint_1` could be `'5mo'` **or** `'10mo'`
depending on the session. If it picks the wrong one, all diff values are negated.  
The new code uses `allele_df["time"].unique()` which preserves the cache-file concatenation
order — always `'5mo'` first when cache files are passed as `[5mo_cache, 10mo_cache]`.

The comparison cell detects this automatically and normalises signs before asserting equality.

In [20]:
# ============================================================
# FULL PRE-REFACTOR allele_freq.py — pasted verbatim
#
# Function names are prefixed with legacy_ to avoid shadowing
# the already-imported new pipeline functions.
# The global variables metadata_dict / DATA_TYPE are renamed to
# legacy_metadata_dict / legacy_DATA_TYPE for the same reason.
#
# Functions NOT redefined here (identical between old and new):
#   load_qc_results       → imported from _allele_freq_common
#   calculate_frequencies → imported from _allele_freq_common
#   filter_constant_positions → imported from allele_freq (unchanged)
#   get_mean_change           → imported from allele_freq (unchanged)
# ============================================================

import argparse
import gc
import logging
import os
import sys
import time
from multiprocessing import Pool, cpu_count

import pandas as pd

setup_logging()

# Module-level globals set by legacy_init_worker (mirrors old init_worker pattern)
legacy_metadata_dict = {}
legacy_DATA_TYPE     = ''

logger = logging.getLogger(__name__)


def legacy_init_worker(metadata, data_type):
    global legacy_metadata_dict, legacy_DATA_TYPE
    legacy_metadata_dict = metadata
    legacy_DATA_TYPE     = data_type


def legacy_process_mag_files(args):
    """Old-style profile loader — keeps ALL columns including MAG_ID, breadth,
    genome_size, average_coverage, ref_base, N.
    The new process_mag_files drops these early (selects only CACHE_COLUMNS_LONGITUDINAL).
    Neither version propagates the extra columns to the final output because
    legacy_calculate_allele_frequency_changes applies an explicit columns_to_keep."""
    sample_id, filepath, mag_id = args

    df = pd.read_csv(filepath, sep='\t', dtype={'gene_id': str})
    df['sample_id'] = sample_id

    metadata_info = legacy_metadata_dict.get(sample_id)
    if metadata_info is None:
        logger.error(f'Sample ID not found for sample {sample_id}')
        return None

    df['group']            = metadata_info['group']
    df['subjectID']        = metadata_info['subjectID']
    df['replicate']        = metadata_info['replicate']
    df['breadth']          = metadata_info['breadth']
    df['genome_size']      = metadata_info['genome_size']
    df['average_coverage'] = metadata_info['average_coverage']

    if legacy_DATA_TYPE == 'longitudinal' and 'time' in metadata_info:
        df['time'] = metadata_info['time']

    df.insert(0, 'MAG_ID', mag_id)        # <-- old pipeline inserted MAG_ID first
    df = calculate_frequencies(df)         # same as new (imported from _allele_freq_common)
    return df


def legacy_save_allele_frequencies(data_dict, output_dir, mag_id):
    """Writes _allele_frequency_longitudinal.tsv.gz — removed in new pipeline
    (Parquet cache replaces it, shared across combinations)."""
    mag_df = pd.concat(
        [df for subject_dict in data_dict.values() for df in subject_dict.values()],
        ignore_index=True,
    )
    os.makedirs(output_dir, exist_ok=True)
    mag_df.to_csv(
        os.path.join(output_dir, f'{mag_id}_allele_frequency_longitudinal.tsv.gz'),
        index=False, sep='\t', compression='gzip',
    )
    logger.info(f'Saved longitudinal TSV for {mag_id} to {output_dir}')


def legacy_create_data_dict(data_list):
    """Organise per-sample DataFrames into {subjectID: {timepoint: df}}.
    Removed in new pipeline — the vectorized merge works directly on the
    concatenated DataFrame without this intermediate structure."""
    data_dict = {}
    for df in data_list:
        subject_ids = df['subjectID'].unique()
        if len(subject_ids) != 1:
            raise ValueError(f"Multiple subjectIDs in DataFrame for sample {df['sample_id'].iloc[0]}")
        subjectID = subject_ids[0]
        timepoints = df['time'].unique()
        if len(timepoints) != 1:
            raise ValueError(f"Multiple timepoints in DataFrame for sample {df['sample_id'].iloc[0]}")
        timepoint = timepoints[0]
        if subjectID not in data_dict:
            data_dict[subjectID] = {}
        data_dict[subjectID][timepoint] = df
    return data_dict


def legacy_calculate_allele_frequency_changes(data_dict, output_dir, mag_id):
    """Per-subject merge loop — the core of what changed in the refactor.

    KEY DIFFERENCES from new calculate_allele_frequency_changes:
      1. Timepoint order: `set()` unpack is NON-DETERMINISTIC — tp1/tp2 assignment
         depends on Python's hash seed. New code uses `unique()` (first-occurrence order).
      2. Algorithm: N sequential pd.merge calls (one per subject). New code does
         ONE global merge with subjectID in the key, equivalent but O(1) merges.
      3. Output format: writes _changes.tsv.gz. New writes _changes.parquet (5–10x faster).
    """
    logger.info('Identifying unique timepoints.')
    unique_timepoints = set()
    for subject_data in data_dict.values():
        unique_timepoints.update(subject_data.keys())
    if len(unique_timepoints) != 2:
        raise ValueError(f'Expected exactly 2 unique timepoints, found {len(unique_timepoints)}.')

    # *** NON-DETERMINISTIC: set has no guaranteed iteration order ***
    timepoint_1, timepoint_2 = unique_timepoints
    logger.info(f'Calculating diffs between {timepoint_1} and {timepoint_2}.')

    subjectIDs_tp1 = {s for s in data_dict if timepoint_1 in data_dict[s]}
    subjectIDs_tp2 = {s for s in data_dict if timepoint_2 in data_dict[s]}
    only1 = subjectIDs_tp1 - subjectIDs_tp2
    only2 = subjectIDs_tp2 - subjectIDs_tp1
    if only1:
        logger.warning(f"SubjectIDs only in '{timepoint_1}': {only1}")
    if only2:
        logger.warning(f"SubjectIDs only in '{timepoint_2}': {only2}")

    common = [s for s in data_dict if timepoint_1 in data_dict[s] and timepoint_2 in data_dict[s]]
    if not common:
        raise ValueError(f'No common subjectIDs between {timepoint_1} and {timepoint_2}.')

    results = []
    for subjectID in common:
        df1 = data_dict[subjectID][timepoint_1]
        df2 = data_dict[subjectID][timepoint_2]
        merged_df = pd.merge(
            df1, df2,
            on=['subjectID', 'contig', 'gene_id', 'position', 'replicate', 'group'],
            suffixes=(f'_{timepoint_1}', f'_{timepoint_2}'),
            how='inner',
        )
        if merged_df.empty:
            logger.warning(f'No matching positions for subjectID {subjectID}.')
            continue
        for nuc in NUCLEOTIDES:
            merged_df[f'{nuc}_diff'] = merged_df[f'{nuc}_{timepoint_2}'] - merged_df[f'{nuc}_{timepoint_1}']
        merged_df['total_coverage_combined'] = (
            merged_df[f'total_coverage_{timepoint_1}'] + merged_df[f'total_coverage_{timepoint_2}']
        )
        columns_to_keep = (
            ['subjectID', 'gene_id', 'contig', 'position', 'replicate', 'group']
            + [f'total_coverage_{timepoint_1}', f'total_coverage_{timepoint_2}', 'total_coverage_combined']
            + [f'{nuc}_{timepoint_1}' for nuc in NUCLEOTIDES]
            + [f'{nuc}_{timepoint_2}' for nuc in NUCLEOTIDES]
            + [f'{nuc}_diff' for nuc in NUCLEOTIDES]
        )
        results.append(merged_df[columns_to_keep])

    if not results:
        logger.error('No allele frequency changes calculated.')
        sys.exit(42)

    allele_changes = pd.concat(results, ignore_index=True)
    # Old pipeline writes .tsv.gz; new pipeline writes .parquet
    allele_changes.to_csv(
        os.path.join(output_dir, f'{mag_id}_allele_frequency_changes.tsv.gz'),
        sep='\t', index=False, compression='gzip',
    )
    return allele_changes


def legacy_process_longitudinal_data(data_list, output_dir, mag_id, disable_filtering):
    """Orchestrates the old longitudinal pipeline.
    Calls legacy_ functions for the steps that changed; calls the shared
    filter_constant_positions and get_mean_change directly (identical code)."""
    data_dict = legacy_create_data_dict(data_list)
    del data_list
    gc.collect()
    legacy_save_allele_frequencies(data_dict, output_dir, mag_id)    # new: no longer written
    allele_changes = legacy_calculate_allele_frequency_changes(data_dict, output_dir, mag_id)
    if not disable_filtering:
        allele_changes = filter_constant_positions(allele_changes, output_dir, mag_id, data_type='longitudinal')
    get_mean_change(allele_changes, mag_id, output_dir)


print('Legacy functions defined.')
print()
print('Shared (unchanged) functions called by both pipelines:')
print('  filter_constant_positions  — imported from allele_freq')
print('  get_mean_change            — imported from allele_freq')
print('  load_qc_results            — imported from _allele_freq_common')
print('  calculate_frequencies      — imported from _allele_freq_common')

Legacy functions defined.

Shared (unchanged) functions called by both pipelines:
  filter_constant_positions  — imported from allele_freq
  get_mean_change            — imported from allele_freq
  load_qc_results            — imported from _allele_freq_common
  calculate_frequencies      — imported from _allele_freq_common


### Run old pipeline end-to-end using legacy functions

In [21]:
# ── Run legacy pipeline end-to-end ───────────────────────────────────────────
legacy_output_dir = tempfile.mkdtemp(prefix='af_full_legacy_')
print(f'Legacy output dir: {legacy_output_dir}')

# 1. Load QC — same function as new pipeline (load_qc_results from _allele_freq_common)
qc_df_leg = load_qc_results(QC_FILE, MAG_ID)

# 2. Build metadata dict — replicated from old main()
metadata_dict_leg = {}
sample_tuples_leg = []
for _, row in qc_df_leg.iterrows():
    sid  = str(row['sample_id'])
    meta = {
        'group':            row['group'],
        'subjectID':        row['subjectID'],
        'replicate':        row['replicate'],
        'breadth':          row['breadth'],
        'genome_size':      row['genome_size'],
        'average_coverage': row['average_coverage'],
    }
    if 'time' in row and pd.notna(row['time']):
        meta['time'] = row['time']
    metadata_dict_leg[sid] = meta
    sample_tuples_leg.append((sid, row['file_path'], MAG_ID))

print(f'Samples: {[t[0] for t in sample_tuples_leg]}')

# 3. Read profiles using legacy_process_mag_files.
#    (Using a single-process loop rather than Pool — identical logic,
#     avoids pickling a notebook-defined function on some platforms.)
legacy_init_worker(metadata_dict_leg, DATA_TYPE)
data_list_leg = [legacy_process_mag_files(t) for t in sample_tuples_leg]
data_list_leg = [d for d in data_list_leg if d is not None]
print(f'\nLoaded {len(data_list_leg)} DataFrames')

# Capture legacy per-sample column schema for the comparison cell
legacy_sample_cols = data_list_leg[0].columns.tolist()
print(f'Columns (legacy, keeps extras): {legacy_sample_cols}')

# 4. Run the full legacy longitudinal pipeline:
#    legacy_create_data_dict → legacy_save_allele_frequencies
#    → legacy_calculate_allele_frequency_changes (set() ordering, per-subject loop, writes .tsv.gz)
#    → filter_constant_positions (shared, unchanged)
#    → get_mean_change (shared, unchanged)
legacy_process_longitudinal_data(data_list_leg, legacy_output_dir, MAG_ID, disable_filtering=False)

print('\nLegacy output files:')
for f in sorted(os.listdir(legacy_output_dir)):
    sz = os.path.getsize(os.path.join(legacy_output_dir, f))
    print(f'  {f}  ({sz/1e6:.1f} MB)')

2026-05-06 14:55:57 [INFO] alleleflux.scripts.analysis._allele_freq_common: Loading QC results from /scratch/gpfs/AMOELLER/sidd/drido/alleleflux_output/alleleflux_output_1/longitudinal/QC/QC_5mo_10mo-1D_AL/MRGM_1066_QC.tsv
2026-05-06 14:55:57 [INFO] alleleflux.scripts.analysis._allele_freq_common: Loaded 3 samples (out of 443 total) that passed coverage and breadth thresholds


Legacy output dir: /tmp/af_full_legacy_p0t0cung
Samples: ['DO_AL_0140_021w', 'DO_AL_0142_044w', 'DO_AL_0140_044w']

Loaded 3 DataFrames
Columns (legacy, keeps extras): ['MAG_ID', 'contig', 'position', 'ref_base', 'total_coverage', 'A', 'C', 'G', 'T', 'N', 'mapq_scores', 'gene_id', 'sample_id', 'group', 'subjectID', 'replicate', 'breadth', 'genome_size', 'average_coverage', 'time', 'A_frequency', 'T_frequency', 'G_frequency', 'C_frequency']


SubjectIDs only in '10mo': {'DO-AL-0142'}
2026-05-06 14:56:28 [INFO] alleleflux.scripts.analysis.allele_freq: Identifying positions where sum of the difference values across all samples for all nucleotides is zero, called zero-diff positions
2026-05-06 14:56:28 [INFO] alleleflux.scripts.analysis.allele_freq: Found 592,819 zero-diff positions.
2026-05-06 14:56:28 [INFO] alleleflux.scripts.analysis.allele_freq: Filtering zero-diff positions
2026-05-06 14:56:28 [INFO] alleleflux.scripts.analysis.allele_freq: Total positions: 594,041, Positions kept: 1,222, Positions removed: 592,819
2026-05-06 14:56:28 [INFO] alleleflux.scripts.analysis.allele_freq: Allele frequency changes with no zero diff positions saved to /tmp/af_full_legacy_p0t0cung/MRGM_1066_allele_frequency_changes_no_zero-diff.tsv.gz
2026-05-06 14:56:28 [INFO] alleleflux.scripts.analysis.allele_freq: Calculating mean changes in allele frequencies for subjectIDs present in the same replicate and group.
2026-05-06 14:56:28 [INFO] a


Legacy output files:
  MRGM_1066_allele_frequency_changes.tsv.gz  (2.5 MB)
  MRGM_1066_allele_frequency_changes_mean.tsv.gz  (0.0 MB)
  MRGM_1066_allele_frequency_changes_no_zero-diff.tsv.gz  (0.0 MB)
  MRGM_1066_allele_frequency_longitudinal.tsv.gz  (10.7 MB)


### Run new pipeline end-to-end (Stage 1 + Stage 2)

In [22]:
# ── New pipeline: Stage 1 (Parquet cache) + Stage 2 (vectorized merge) ───────
new_full_output_dir = tempfile.mkdtemp(prefix='af_full_new_')
new_full_cache_dir  = os.path.join(new_full_output_dir, 'cache')
new_full_analysis   = os.path.join(new_full_output_dir, 'analysis')
os.makedirs(new_full_cache_dir)
os.makedirs(new_full_analysis)
print(f'New pipeline output dir: {new_full_output_dir}')

# Stage 1: write one Parquet cache per timepoint
cache_paths_full = {}
for tp in TIMEPOINTS:
    qc_tp    = filter_qc_to_timepoint(QC_FILE, MAG_ID, tp)
    meta_tp, base_tp = build_metadata_from_qc(qc_tp)
    tuples_tp = [(sid, fp, MAG_ID) for sid, fp in base_tp]
    _cm.metadata_dict = meta_tp
    _cm.DATA_TYPE     = DATA_TYPE
    dfs = [process_mag_files(t) for t in tuples_tp]
    dfs = [d for d in dfs if d is not None]
    combined = pd.concat(dfs, ignore_index=True)
    path = os.path.join(new_full_cache_dir, f'{MAG_ID}_{GR_COMBO}_{tp}_allele_frequency.parquet')
    combined.to_parquet(path, engine='pyarrow', compression='snappy', index=False)
    cache_paths_full[tp] = path
    print(f'  Stage 1 tp={tp}: {len(combined):,} rows → {os.path.basename(path)}')

# Capture new pipeline's intermediate per-sample column schema for comparison
new_sample_cols = combined.columns.tolist()

# Stage 2: load Parquet, group filter, vectorized merge, filter, mean
allele_df_full = load_cache_files([cache_paths_full[tp] for tp in TIMEPOINTS])
allele_df_full = allele_df_full[allele_df_full['group'].isin(GROUPS)].copy()

new_changes_full = calculate_allele_frequency_changes(allele_df_full, new_full_analysis, MAG_ID)
new_nozero_full  = filter_constant_positions(new_changes_full, new_full_analysis, MAG_ID, data_type='longitudinal')
new_mean_full    = get_mean_change(new_nozero_full, MAG_ID, new_full_analysis)

print(f'\nNew pipeline _changes shape:      {new_changes_full.shape}')
print(f'New pipeline _no_zero-diff shape: {new_nozero_full.shape}')
print(f'New pipeline _changes_mean shape: {new_mean_full.shape}')
print(f'New pipeline output files:')
for f in sorted(os.listdir(new_full_analysis)):
    sz = os.path.getsize(os.path.join(new_full_analysis, f))
    print(f'  {f}  ({sz/1e6:.1f} MB)')

2026-05-06 14:56:28 [INFO] alleleflux.scripts.analysis._allele_freq_common: Loading QC results from /scratch/gpfs/AMOELLER/sidd/drido/alleleflux_output/alleleflux_output_1/longitudinal/QC/QC_5mo_10mo-1D_AL/MRGM_1066_QC.tsv
2026-05-06 14:56:28 [INFO] alleleflux.scripts.analysis._allele_freq_common: Loaded 3 samples (out of 443 total) that passed coverage and breadth thresholds
2026-05-06 14:56:28 [INFO] alleleflux.scripts.analysis.allele_freq_cache: Found 1 samples for MAG MRGM_1066 at timepoint '5mo'.


New pipeline output dir: /tmp/af_full_new_yr5dh73k


2026-05-06 14:56:29 [INFO] alleleflux.scripts.analysis._allele_freq_common: Loading QC results from /scratch/gpfs/AMOELLER/sidd/drido/alleleflux_output/alleleflux_output_1/longitudinal/QC/QC_5mo_10mo-1D_AL/MRGM_1066_QC.tsv
2026-05-06 14:56:29 [INFO] alleleflux.scripts.analysis._allele_freq_common: Loaded 3 samples (out of 443 total) that passed coverage and breadth thresholds
2026-05-06 14:56:29 [INFO] alleleflux.scripts.analysis.allele_freq_cache: Found 2 samples for MAG MRGM_1066 at timepoint '10mo'.


  Stage 1 tp=5mo: 681,573 rows → MRGM_1066_1D_AL_5mo_allele_frequency.parquet


2026-05-06 14:56:31 [INFO] alleleflux.scripts.analysis.allele_freq: Reading cache file /tmp/af_full_new_yr5dh73k/cache/MRGM_1066_1D_AL_5mo_allele_frequency.parquet
2026-05-06 14:56:31 [INFO] alleleflux.scripts.analysis.allele_freq: Reading cache file /tmp/af_full_new_yr5dh73k/cache/MRGM_1066_1D_AL_10mo_allele_frequency.parquet


  Stage 1 tp=10mo: 1,824,263 rows → MRGM_1066_1D_AL_10mo_allele_frequency.parquet


2026-05-06 14:56:31 [INFO] alleleflux.scripts.analysis.allele_freq: Loaded 2,505,836 rows from 2 cache file(s).
2026-05-06 14:56:32 [INFO] alleleflux.scripts.analysis.allele_freq: Calculating allele frequency changes between 5mo and 10mo.
2026-05-06 14:56:32 [WARNING] alleleflux.scripts.analysis.allele_freq: SubjectIDs only in '10mo' (no match at '5mo'): {'DO-AL-0142'}
2026-05-06 14:56:34 [INFO] alleleflux.scripts.analysis.allele_freq: Allele frequency changes saved to /tmp/af_full_new_yr5dh73k/analysis/MRGM_1066_allele_frequency_changes.parquet
2026-05-06 14:56:34 [INFO] alleleflux.scripts.analysis.allele_freq: Identifying positions where sum of the difference values across all samples for all nucleotides is zero, called zero-diff positions
2026-05-06 14:56:34 [INFO] alleleflux.scripts.analysis.allele_freq: Found 592,819 zero-diff positions.
2026-05-06 14:56:34 [INFO] alleleflux.scripts.analysis.allele_freq: Filtering zero-diff positions
2026-05-06 14:56:34 [INFO] alleleflux.scripts.a


New pipeline _changes shape:      (594041, 21)
New pipeline _no_zero-diff shape: (1222, 21)
New pipeline _changes_mean shape: (1222, 10)
New pipeline output files:
  MRGM_1066_allele_frequency_changes.parquet  (4.4 MB)
  MRGM_1066_allele_frequency_changes_mean.tsv.gz  (0.0 MB)
  MRGM_1066_allele_frequency_changes_no_zero-diff.tsv.gz  (0.0 MB)


### Compare all outputs — detect and handle timepoint sign flip

The column names in `_changes.tsv.gz` reveal which timepoint the legacy `set()` unpack
assigned as `tp1`. If it differs from the new code's order (`5mo` first), the diff columns
are negated and the per-timepoint column names are swapped before asserting equality.

The comparison also prints the intermediate per-sample DataFrame schemas to confirm
that extra columns kept by the legacy pipeline (`MAG_ID`, `breadth`, `genome_size`,
`average_coverage`) do not appear in the final outputs.

In [23]:
# ── Detect which timepoint legacy code called tp1 ────────────────────────────
# The column names in _changes.tsv.gz reveal the set() unpack order.
# e.g. 'total_coverage_5mo' → tp1='5mo', 'total_coverage_10mo' → tp1='10mo'
legacy_changes = pd.read_csv(
    os.path.join(legacy_output_dir, f'{MAG_ID}_allele_frequency_changes.tsv.gz'), sep='\t'
)
cov_cols   = [c for c in legacy_changes.columns if c.startswith('total_coverage_') and c != 'total_coverage_combined']
legacy_tp1 = cov_cols[0].replace('total_coverage_', '')
legacy_tp2 = cov_cols[1].replace('total_coverage_', '')
new_tp1, new_tp2 = list(TIMEPOINTS)

print('─' * 60)
print(f'Legacy code  set()    → tp1={legacy_tp1!r}  tp2={legacy_tp2!r}')
print(f'New code     unique() → tp1={new_tp1!r}  tp2={new_tp2!r}')
print()

rename_map = {}
if legacy_tp1 != new_tp1:
    print('[NOTE] Timepoint ordering DIFFERS — legacy diffs are negated relative to new.')
    print('       This is a non-determinism bug in the old set() unpack.')
    print('       Normalising signs and column names before comparison...')
    # Flip diff signs
    for nuc in NUCLEOTIDES:
        legacy_changes[f'{nuc}_diff'] = -legacy_changes[f'{nuc}_diff']
    # Rename per-timepoint columns so both DataFrames use the same names
    for nuc in NUCLEOTIDES:
        rename_map[f'{nuc}_{legacy_tp1}'] = f'{nuc}_{new_tp1}'
        rename_map[f'{nuc}_{legacy_tp2}'] = f'{nuc}_{new_tp2}'
    rename_map[f'total_coverage_{legacy_tp1}'] = f'total_coverage_{new_tp1}'
    rename_map[f'total_coverage_{legacy_tp2}'] = f'total_coverage_{new_tp2}'
    legacy_changes.rename(columns=rename_map, inplace=True)
else:
    print('[OK] Timepoint ordering matches — diffs have the same sign.')

# ── Load remaining legacy outputs ─────────────────────────────────────────────
legacy_nozero = pd.read_csv(
    os.path.join(legacy_output_dir, f'{MAG_ID}_allele_frequency_changes_no_zero-diff.tsv.gz'), sep='\t'
)
legacy_mean = pd.read_csv(
    os.path.join(legacy_output_dir, f'{MAG_ID}_allele_frequency_changes_mean.tsv.gz'), sep='\t'
)
# Propagate normalisation to no_zero if needed
if rename_map:
    legacy_nozero.rename(columns=rename_map, inplace=True)
    for nuc in NUCLEOTIDES:
        legacy_nozero[f'{nuc}_diff'] = -legacy_nozero[f'{nuc}_diff']
    legacy_mean[[f'{nuc}_diff_mean' for nuc in NUCLEOTIDES]] *= -1

# ── Compare all three output tables ──────────────────────────────────────────
pairs = [
    (legacy_changes, new_changes_full,  SORT_CHANGES, '_changes'),
    (legacy_nozero,  new_nozero_full,   SORT_CHANGES, '_no_zero-diff'),
    (legacy_mean,    new_mean_full,     SORT_MEAN,    '_changes_mean'),
]
all_passed = True
for old_df, new_df, key, label in pairs:
    passed = compare_outputs(old_df.copy(), new_df.copy(), key, label)
    all_passed = all_passed and passed

# ── Schema comparison: extra columns in legacy intermediate data ──────────────
print('\n── Column schema: per-sample DataFrame (intermediate) ──────────────')
print(f'  Legacy (process_mag_files keeps extras): {legacy_sample_cols}')
print(f'  New    (process_mag_files drops extras) : {new_sample_cols}')
print(f'  Dropped by new pipeline: {set(legacy_sample_cols) - set(new_sample_cols)}')
print(f'  Both   (final output columns identical): {list(new_changes_full.columns)}')

# ── Legacy-only output: longitudinal TSV ─────────────────────────────────────
long_tsv = os.path.join(legacy_output_dir, f'{MAG_ID}_allele_frequency_longitudinal.tsv.gz')
if os.path.exists(long_tsv):
    long_df = pd.read_csv(long_tsv, sep='\t')
    print(f'\n── Legacy-only _longitudinal.tsv.gz ────────────────────────')
    print(f'  Shape: {long_df.shape}')
    print(f'  Columns: {long_df.columns.tolist()}')
    print(f'  (new pipeline does not write this file — Parquet cache replaces it)')

shutil.rmtree(legacy_output_dir, ignore_errors=True)
shutil.rmtree(new_full_output_dir, ignore_errors=True)

print()
print('=' * 60)
print(f'FULL CODE COMPARISON: {"ALL PASSED" if all_passed else "SOME FAILED"}')
if all_passed:
    print()
    print('Confirmed: old and new pipelines are mathematically equivalent.')
    print('The refactor changed:')
    print('  1. Input format   — QC TSV + profile TSVs  →  Parquet cache')
    print('  2. Intermediate   — extra cols kept         →  dropped early (no effect on output)')
    print('  3. Timepoint order— set() (non-deterministic)→ unique() (stable)')
    print('  4. Diff algorithm — per-subject loop        →  single global merge')
    print('  5. Output format  — _changes.tsv.gz         →  _changes.parquet (faster write)')
    print('  6. Extra output   — _longitudinal.tsv.gz    →  removed (cache replaces it)')

────────────────────────────────────────────────────────────
Legacy code  set()    → tp1='5mo'  tp2='10mo'
New code     unique() → tp1='5mo'  tp2='10mo'

[OK] Timepoint ordering matches — diffs have the same sign.

── _changes ────────────────────────────
  Old: (594041, 21)   New: (594041, 21)


/tmp/ipykernel_403099/2777620492.py:23: FutureWarning: Mismatched null-like values nan and None found. In a future version, pandas equality-testing functions (e.g. assert_frame_equal) will consider these not-matching and raise.
  pd.testing.assert_frame_equal(old_s, new_s, check_like=False, check_dtype=False, atol=atol, rtol=0)


  [PASS] 594,041 rows match exactly (atol=1e-12)

── _no_zero-diff ────────────────────────────
  Old: (1222, 21)   New: (1222, 21)
  [PASS] 1,222 rows match exactly (atol=1e-12)

── _changes_mean ────────────────────────────
  Old: (1222, 10)   New: (1222, 10)
  [PASS] 1,222 rows match exactly (atol=1e-12)

── Column schema: per-sample DataFrame (intermediate) ──────────────
  Legacy (process_mag_files keeps extras): ['MAG_ID', 'contig', 'position', 'ref_base', 'total_coverage', 'A', 'C', 'G', 'T', 'N', 'mapq_scores', 'gene_id', 'sample_id', 'group', 'subjectID', 'replicate', 'breadth', 'genome_size', 'average_coverage', 'time', 'A_frequency', 'T_frequency', 'G_frequency', 'C_frequency']
  New    (process_mag_files drops extras) : ['contig', 'position', 'gene_id', 'total_coverage', 'A', 'C', 'G', 'T', 'sample_id', 'group', 'subjectID', 'replicate', 'A_frequency', 'T_frequency', 'G_frequency', 'C_frequency', 'time']
  Dropped by new pipeline: {'N', 'mapq_scores', 'genome_size', 'ref_ba

/tmp/ipykernel_403099/2777620492.py:23: FutureWarning: Mismatched null-like values nan and None found. In a future version, pandas equality-testing functions (e.g. assert_frame_equal) will consider these not-matching and raise.
  pd.testing.assert_frame_equal(old_s, new_s, check_like=False, check_dtype=False, atol=atol, rtol=0)
/tmp/ipykernel_403099/1246482896.py:70: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  long_df = pd.read_csv(long_tsv, sep='\t')



── Legacy-only _longitudinal.tsv.gz ────────────────────────
  Shape: (2505836, 24)
  Columns: ['MAG_ID', 'contig', 'position', 'ref_base', 'total_coverage', 'A', 'C', 'G', 'T', 'N', 'mapq_scores', 'gene_id', 'sample_id', 'group', 'subjectID', 'replicate', 'breadth', 'genome_size', 'average_coverage', 'time', 'A_frequency', 'T_frequency', 'G_frequency', 'C_frequency']
  (new pipeline does not write this file — Parquet cache replaces it)

FULL CODE COMPARISON: ALL PASSED

Confirmed: old and new pipelines are mathematically equivalent.
The refactor changed:
  1. Input format   — QC TSV + profile TSVs  →  Parquet cache
  2. Intermediate   — extra cols kept         →  dropped early (no effect on output)
  3. Timepoint order— set() (non-deterministic)→ unique() (stable)
  4. Diff algorithm — per-subject loop        →  single global merge
  5. Output format  — _changes.tsv.gz         →  _changes.parquet (faster write)
  6. Extra output   — _longitudinal.tsv.gz    →  removed (cache replaces 